In [3]:
############################################################
# STEP3.2_04_summary.R
#
# - 入力: STEP3.2_03_clustering.R の出力
#     sub/03_clustering_STEP3.2_signlessCorr/run_<run_id>/<unit>/<dataset>/
#       labels/ClusterAssign_*.csv
#       indices/k_score_*.csv
#
# - 出力: サマリー
#     sub/04_summary_STEP3.2_signlessCorr/run_<run_id>/<unit>/
#       cluster_labels_matrix_<unit>_<dataset>_<run_id>.csv
#       cluster_k_summary_<unit>_<dataset>_<run_id>.csv
#       ARI_top3_vs_cumeig_linear_<unit>_<dataset>_<run_id>.csv
#       ARI_top3_vs_cumeig_nonlinear_<unit>_<dataset>_<run_id>.csv
############################################################

## パッケージ読み込み -------------------------------------------------------
if (!require(mclust))  { install.packages("mclust");  library(mclust) }  # ARI
if (!require(dplyr))   { install.packages("dplyr");   library(dplyr)  }
if (!require(stringr)) { install.packages("stringr"); library(stringr) }

set.seed(42)

############################################################
# 0. パス設定 & 基本設定
############################################################

root_base <- "/Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127"
sub_base  <- file.path(root_base, "sub")

step03_base <- file.path(sub_base, "03_clustering_STEP3.2_signlessCorr")
step04_base <- file.path(sub_base, "04_summary_STEP3.2_signlessCorr")
dir.create(step04_base, showWarnings = FALSE, recursive = TRUE)

# 対象データセット
datasets_variables <- c("OH", "FP")
datasets_samples   <- c("A_OH_plus_others",
                        "B_FP_plus_others",
                        "C_OH_only",
                        "D_FP_only")

# NbClust 指標
index_list <- c("silhouette","dunn","gap","ch","db","ptbiserial")

# index の最適化方向
maximize_indices <- c("silhouette","dunn","gap","ch","ptbiserial")
minimize_indices <- c("db")

############################################################
# 1. run_id の取得
############################################################

# ★ Jupyter から使うときはここを書き換える：
#    03_clustering で使用した run_id を指定する
# 例: override_run_id <- "20251128_193916"
override_run_id <- "20251130_153500"

get_run_id <- function() {
  # 1) override_run_id があればそれを最優先
  if (!is.na(override_run_id) &&
      is.character(override_run_id) &&
      grepl("^\\d{8}_\\d{6}$", override_run_id)) {
    message("[INFO] Using override_run_id = ", override_run_id)
    return(override_run_id)
  }
  # 2) Rscript 実行時の引数 (Rscript STEP3.2_04_summary.R 20251128_193916)
  args <- commandArgs(trailingOnly = TRUE)
  if (length(args) >= 1 && grepl("^\\d{8}_\\d{6}$", args[1])) {
    return(args[1])
  }
  # 3) どちらも無い場合はエラーにする
  stop("Usage: Rscript STEP3.2_04_summary.R <run_id (YYYYMMDD_HHMMSS)>\n",
       "  - or set override_run_id <- \"YYYYMMDD_HHMMSS\" at the top of this script.")
}

run_id <- get_run_id()
cat(">>> STEP3.2_04_summary — run_id:", run_id, "\n")

step03_run_dir <- file.path(step03_base, paste0("run_", run_id))
step04_run_dir <- file.path(step04_base, paste0("run_", run_id))
dir.create(step04_run_dir, showWarnings = FALSE, recursive = TRUE)

cat(">>> STEP3.2_04_summary input dir:  ", normalizePath(step03_run_dir), "\n")
cat(">>> STEP3.2_04_summary output dir: ", normalizePath(step04_run_dir), "\n")

############################################################
# 2. 1データセットのサマリー処理
############################################################

process_one_dataset_summary <- function(unit, dataset) {
  message("\n[STEP3.2_04] unit = ", unit, ", dataset = ", dataset)
  
  # 03_clustering の出力ディレクトリ
  clu_dir_ds <- file.path(step03_run_dir, unit, dataset)
  if (!dir.exists(clu_dir_ds)) {
    stop("[ERROR] Clustering dir not found: ", clu_dir_ds)
  }
  labels_dir <- file.path(clu_dir_ds, "labels")
  idx_dir    <- file.path(clu_dir_ds, "indices")
  
  if (!dir.exists(labels_dir)) {
    stop("[ERROR] labels dir not found: ", labels_dir)
  }
  if (!dir.exists(idx_dir)) {
    warning("[WARN] indices dir not found: ", idx_dir,
            " (k-summary は欠損になる可能性があります)")
  }
  
  # 04_summary 出力ディレクトリ
  out_unit_dir <- file.path(step04_run_dir, unit)
  dir.create(out_unit_dir, showWarnings = FALSE, recursive = TRUE)
  
  ##########################################################
  # 2-1. cluster_labels_matrix の作成
  ##########################################################
  
  # mode_dim: "linear_top3", "linear_cumeig", "nonlinear_top3", "nonlinear_cumeig"
  mode_dims <- c("linear_top3", "linear_cumeig",
                 "nonlinear_top3", "nonlinear_cumeig")
  
  labels_df <- NULL
  
  # NbClust ラベル
  for (mode_dim in mode_dims) {
    parts    <- strsplit(mode_dim, "_")[[1]]
    mode     <- parts[1]   # "linear" or "nonlinear"
    dim_mode <- parts[2]   # "top3" or "cumeig"
    cond_tag <- sprintf("%s_%s", mode, dim_mode)
    
    for (idx in index_list) {
      lab_file <- file.path(
        labels_dir,
        sprintf("ClusterAssign_%s_%s_%s_%s_%s_%s.csv",
                mode, dim_mode, idx, unit, dataset, run_id)
      )
      if (!file.exists(lab_file)) next
      
      df_lab <- read.csv(lab_file, stringsAsFactors = FALSE)
      if (!all(c("ID","Cluster") %in% colnames(df_lab))) {
        warning("[WARN] Unexpected columns in ", lab_file,
                " (skip this file).")
        next
      }
      col_name <- sprintf("%s_%s", cond_tag, idx)  # 例: linear_top3_silhouette
      
      tmp <- df_lab[, c("ID","Cluster")]
      names(tmp)[2] <- col_name
      
      if (is.null(labels_df)) {
        labels_df <- tmp
      } else {
        labels_df <- merge(labels_df, tmp, by = "ID", all = TRUE)
      }
    }
  }
  
  # k固定のラベル (10, 25, 50)
  for (mode_dim in mode_dims) {
    parts    <- strsplit(mode_dim, "_")[[1]]
    mode     <- parts[1]
    dim_mode <- parts[2]
    
    for (k_fixed in c(10, 25, 50)) {
      lab_file_k <- file.path(
        labels_dir,
        sprintf("ClusterAssign_%s_%s_k%d_%s_%s_%s.csv",
                mode, dim_mode, k_fixed, unit, dataset, run_id)
      )
      if (!file.exists(lab_file_k)) next
      
      df_lab_k <- read.csv(lab_file_k, stringsAsFactors = FALSE)
      if (!all(c("ID","Cluster") %in% colnames(df_lab_k))) {
        warning("[WARN] Unexpected columns in ", lab_file_k,
                " (skip this file).")
        next
      }
      col_name_k <- sprintf("%s_%s_k%d", mode, dim_mode, k_fixed)
      
      tmp_k <- df_lab_k[, c("ID","Cluster")]
      names(tmp_k)[2] <- col_name_k
      
      if (is.null(labels_df)) {
        labels_df <- tmp_k
      } else {
        labels_df <- merge(labels_df, tmp_k, by = "ID", all = TRUE)
      }
    }
  }
  
  if (is.null(labels_df)) {
    warning("[WARN] No cluster labels found for unit = ", unit,
            ", dataset = ", dataset, ". Skip summary.")
    return(invisible(NULL))
  }
  
  # ID順にソート
  labels_df <- labels_df[order(labels_df$ID), ]
  
  # cluster_labels_matrix_<unit>_<dataset>_<run_id>.csv を保存
  labels_file <- file.path(
    out_unit_dir,
    sprintf("cluster_labels_matrix_%s_%s_%s.csv", unit, dataset, run_id)
  )
  write.csv(labels_df, labels_file, row.names = FALSE)
  message("  [SAVED] cluster_labels_matrix: ", labels_file)
  
  ##########################################################
  # 2-2. indexごとのベストk一覧 (cluster_k_summary)
  ##########################################################
  
  ksum <- data.frame(
    mode    = character(0),
    dimmode = character(0),
    index   = character(0),
    k       = integer(0),
    stringsAsFactors = FALSE
  )
  
  for (mode_dim in mode_dims) {
    parts    <- strsplit(mode_dim, "_")[[1]]
    mode     <- parts[1]
    dim_mode <- parts[2]
    cond_tag <- sprintf("%s_%s", mode, dim_mode)
    
    for (idx in index_list) {
      kscore_file <- file.path(
        idx_dir,
        sprintf("k_score_%s_%s_%s_%s_%s.csv",
                cond_tag, unit, dataset, idx, run_id)
      )
      if (!file.exists(kscore_file)) next
      
      df_k <- read.csv(kscore_file, stringsAsFactors = FALSE)
      if (!all(c("k","score") %in% colnames(df_k))) {
        warning("[WARN] Unexpected columns in ", kscore_file,
                " (skip this file).")
        next
      }
      
      # score が NA のものを除外
      df_k <- df_k[!is.na(df_k$score), ]
      if (nrow(df_k) == 0) {
        best_k <- NA_integer_
      } else {
        if (idx %in% maximize_indices) {
          best_row <- which.max(df_k$score)
        } else if (idx %in% minimize_indices) {
          best_row <- which.min(df_k$score)
        } else {
          # 想定外の場合はいちおう最大化
          best_row <- which.max(df_k$score)
        }
        best_k <- as.integer(df_k$k[best_row])
      }
      
      ksum <- rbind(ksum, data.frame(
        mode    = mode,
        dimmode = dim_mode,
        index   = idx,
        k       = best_k,
        stringsAsFactors = FALSE
      ))
    }
  }
  
  ksum_file <- file.path(
    out_unit_dir,
    sprintf("cluster_k_summary_%s_%s_%s.csv", unit, dataset, run_id)
  )
  write.csv(ksum, ksum_file, row.names = FALSE)
  message("  [SAVED] cluster_k_summary: ", ksum_file)
  
  ##########################################################
  # 2-3. ARI (top3 vs cumeig, linear / nonlinear)
  ##########################################################
  
  # labels_df を使って ARI を計算
  for (mode in c("linear","nonlinear")) {
    ari_rows <- lapply(index_list, function(ix) {
      col_top3   <- sprintf("%s_top3_%s", mode, ix)
      col_cumeig <- sprintf("%s_cumeig_%s", mode, ix)
      
      if (!(col_top3 %in% colnames(labels_df)) ||
          !(col_cumeig %in% colnames(labels_df))) {
        return(data.frame(
          Index = ix,
          ARI_top3_vs_cumeig = NA_real_,
          stringsAsFactors = FALSE
        ))
      }
      a <- labels_df[[col_top3]]
      b <- labels_df[[col_cumeig]]
      if (all(is.na(a)) || all(is.na(b))) {
        return(data.frame(
          Index = ix,
          ARI_top3_vs_cumeig = NA_real_,
          stringsAsFactors = FALSE
        ))
      }
      data.frame(
        Index = ix,
        ARI_top3_vs_cumeig = mclust::adjustedRandIndex(
          as.factor(a), as.factor(b)
        ),
        stringsAsFactors = FALSE
      )
    })
    
    df_ari <- dplyr::bind_rows(ari_rows)
    ari_file <- file.path(
      out_unit_dir,
      sprintf("ARI_top3_vs_cumeig_%s_%s_%s_%s.csv",
              mode, unit, dataset, run_id)
    )
    write.csv(df_ari, ari_file, row.names = FALSE)
    message("  [SAVED] ARI (", mode, "): ", ari_file)
  }
  
  message("[DONE] Summary finished for unit = ", unit,
          ", dataset = ", dataset)
}

############################################################
# 3. メインループ
############################################################

cat("\n===== STEP3.2_04_summary: Variables (OH/FP) =====\n")
for (ds in datasets_variables) {
  process_one_dataset_summary("variables", ds)
}

cat("\n===== STEP3.2_04_summary: Samples (A/B/C/D) =====\n")
for (ds in datasets_samples) {
  process_one_dataset_summary("samples", ds)
}

cat("\n✅ STEP3.2_04_summary finished. run_id = ", run_id, "\n")
cat("   Output: ", normalizePath(step04_run_dir), "\n")


[INFO] Using override_run_id = 20251130_153500



>>> STEP3.2_04_summary <U+2014> run_id: 20251130_153500 
>>> STEP3.2_04_summary input dir:   /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127/sub/03_clustering_STEP3.2_signlessCorr/run_20251130_153500 
>>> STEP3.2_04_summary output dir:  /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127/sub/04_summary_STEP3.2_signlessCorr/run_20251130_153500 

===== STEP3.2_04_summary: Variables (OH/FP) =====



[STEP3.2_04] unit = variables, dataset = OH

  [SAVED] cluster_labels_matrix: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127/sub/04_summary_STEP3.2_signlessCorr/run_20251130_153500/variables/cluster_labels_matrix_variables_OH_20251130_153500.csv

  [SAVED] cluster_k_summary: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127/sub/04_summary_STEP3.2_signlessCorr/run_20251130_153500/variables/cluster_k_summary_variables_OH_20251130_153500.csv

  [SAVED] ARI (linear): /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127/sub/04_summary_STEP3.2_signlessCorr/run_20251130_153500/variables/ARI_top3_vs_cumeig_linear_variables_OH_20251130_153500.csv

  [SAVED] ARI (nonlinear): /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_


===== STEP3.2_04_summary: Samples (A/B/C/D) =====



[STEP3.2_04] unit = samples, dataset = A_OH_plus_others

  [SAVED] cluster_labels_matrix: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127/sub/04_summary_STEP3.2_signlessCorr/run_20251130_153500/samples/cluster_labels_matrix_samples_A_OH_plus_others_20251130_153500.csv

  [SAVED] cluster_k_summary: /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127/sub/04_summary_STEP3.2_signlessCorr/run_20251130_153500/samples/cluster_k_summary_samples_A_OH_plus_others_20251130_153500.csv

  [SAVED] ARI (linear): /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127/sub/04_summary_STEP3.2_signlessCorr/run_20251130_153500/samples/ARI_top3_vs_cumeig_linear_samples_A_OH_plus_others_20251130_153500.csv

  [SAVED] ARI (nonlinear): /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_clu


<U+2705> STEP3.2_04_summary finished. run_id =  20251130_153500 
   Output:  /Volumes/csbdeep11/_yasu-i/20250303_rebuiled/cal_cluster_No/20251026_under_25clusters_program_and_result/for_checking_20251127/sub/04_summary_STEP3.2_signlessCorr/run_20251130_153500 
